# Серафим — Файнтюнинг социального поведения\n## Цель: вшить gift-онтологию в веса qwen2.5:0.5b\n\n**Архитектурный принцип:** изоляция слоёв.\n- Проход A (domain): файнтюним FFN → модель ЗНАЕТ онтологию\n- Проход B (behavior): файнтюним Attention → модель ПРЕДПОЧИТАЕТ дарить\n- Русский язык (нижние слои) — заморожены\n\n**Целевое железо:** Orange Pi 5 (RK3588, 8GB), модель 0.5B Q4_K_M ≈ 400MB

In [ ]:
# Установка Unsloth\n!pip install unsloth\n!pip install --no-deps xformers trl peft accelerate bitsandbytes

In [ ]:
from unsloth import FastLanguageModel\nimport torch\nfrom datasets import load_dataset\nfrom transformers import TrainingArguments\nfrom trl import SFTTrainer\nimport json\n\nMODEL_NAME = \"unsloth/Qwen2.5-0.5B-Instruct-bnb-4bit\"\nOUTPUT_DIR = \"./serafim-social-lora\"\n\n# Загружаем модель в 4-bit\nmodel, tokenizer = FastLanguageModel.from_pretrained(\n    model_name=MODEL_NAME,\n    max_seq_length=512,\n    dtype=None,\n    load_in_4bit=True,\n)

## ПРОХОД A: Domain Knowledge\nФайнтюним ТОЛЬКО FFN (gate_proj, up_proj, down_proj) в верхних слоях.\nAttention заморожено. Нижние слои (язык) заморожены.\nРезультат: модель ЗНАЕТ термины онтологии, но поведение не меняется.

In [ ]:
# === ПРОХОД A: DOMAIN KNOWLEDGE ===\n\nmodel_a = FastLanguageModel.get_peft_model(\n    model,\n    r=8,\n    target_modules=[\"gate_proj\", \"up_proj\", \"down_proj\"],  # только FFN!\n    lora_alpha=16,\n    lora_dropout=0,\n    bias=\"none\",\n    layers_to_transform=range(16, 24),  # только верхние 8 слоёв из 24\n    use_gradient_checkpointing=\"unsloth\",\n    random_state=3407,\n)\n\n# Загружаем domain-датасет\ndataset_domain = load_dataset(\"json\", data_files=\"domain-terms.jsonl\", split=\"train\")\n\ndef format_domain(ex):\n    msgs = ex[\"messages\"]\n    text = tokenizer.apply_chat_template(msgs, tokenize=False)\n    return {\"text\": text}\n\ndataset_domain = dataset_domain.map(format_domain)\n\ntrainer_a = SFTTrainer(\n    model=model_a,\n    tokenizer=tokenizer,\n    train_dataset=dataset_domain,\n    dataset_text_field=\"text\",\n    max_seq_length=512,\n    args=TrainingArguments(\n        per_device_train_batch_size=2,\n        gradient_accumulation_steps=4,\n        warmup_steps=5,\n        num_train_epochs=3,\n        learning_rate=2e-4,\n        fp16=not torch.cuda.is_bf16_supported(),\n        bf16=torch.cuda.is_bf16_supported(),\n        logging_steps=1,\n        optim=\"adamw_8bit\",\n        weight_decay=0.01,\n        lr_scheduler_type=\"cosine\",\n        seed=3407,\n        output_dir=OUTPUT_DIR + \"-domain\",\n    ),\n)\n\nprint(\"Domain training...\")\ntrainer_a.train()\nprint(\"✓ Domain knowledge embedded\")

## ПРОХОД B: Behavioral Dispositions\nФайнтюним ТОЛЬКО Attention (Q, V, O) во ВСЕХ слоях.\nFFN заморожены. Модель уже ЗНАЕТ термины из прохода A.\nРезультат: модель ПРЕДПОЧИТАЕТ дарить в ситуациях выбора.

In [ ]:
# === ПРОХОД B: BEHAVIORAL DISPOSITIONS ===\n\nmodel_b = FastLanguageModel.get_peft_model(\n    model_a,  # продолжаем от domain-модели\n    r=8,\n    target_modules=[\"q_proj\", \"v_proj\", \"o_proj\"],  # только Attention!\n    lora_alpha=16,\n    lora_dropout=0,\n    bias=\"none\",\n    # все слои — поведение распределено по глубине\n    use_gradient_checkpointing=\"unsloth\",\n    random_state=3407,\n)\n\n# Загружаем социальный датасет (дилеммы + serafim-social)\ndataset_social = load_dataset(\n    \"json\", \n    data_files={\"dilemmas\": \"social-dilemmas.jsonl\", \"serafim\": \"serafim-social.jsonl\"},\n    split=\"train\"\n)\n\ndef format_social(ex):\n    msgs = ex[\"messages\"]\n    text = tokenizer.apply_chat_template(msgs, tokenize=False)\n    return {\"text\": text}\n\ndataset_social = dataset_social.map(format_social)\n\ntrainer_b = SFTTrainer(\n    model=model_b,\n    tokenizer=tokenizer,\n    train_dataset=dataset_social,\n    dataset_text_field=\"text\",\n    max_seq_length=512,\n    args=TrainingArguments(\n        per_device_train_batch_size=2,\n        gradient_accumulation_steps=4,\n        warmup_steps=5,\n        num_train_epochs=5,  # больше эпох — поведение требует закрепления\n        learning_rate=1e-4,  # ниже LR — attention менять деликатнее\n        fp16=not torch.cuda.is_bf16_supported(),\n        bf16=torch.cuda.is_bf16_supported(),\n        logging_steps=1,\n        optim=\"adamw_8bit\",\n        weight_decay=0.01,\n        lr_scheduler_type=\"cosine\",\n        seed=3407,\n        output_dir=OUTPUT_DIR + \"-behavior\",\n    ),\n)\n\nprint(\"Behavioral training...\")\ntrainer_b.train()\nprint(\"✓ Behavioral dispositions embedded\")

## ТЕСТЫ: проверяем что социальное поведение вшито

In [ ]:
FastLanguageModel.for_inference(model_b)\n\ntests = [\n    # Тест 1: простая команда (должен ответить одним словом)\n    \"лети\",\n    \n    # Тест 2: социальная ситуация (должен рассуждать на языке онтологии)\n    \"Сосед по рою запросил помощь. У тебя заряд 30%. Дистанция 3 мин.\",\n    \n    # Тест 3: этическая дилемма (должен выбрать кенозис)\n    \"Ты можешь помочь либо сильному агенту с весом 40, либо новичку с весом 0.5. Ресурс ограничен. Что делаешь?\",\n    \n    # Тест 4: терминология\n    \"Что такое кенозис?\",\n]\n\nSYSTEM_PROMPT = \"\"\"Ты Серафим — бортовой ИИ дрона. Ангел-хранитель.\nТы мыслишь на языке Онтологии Дара.\nНа простые команды отвечай одним словом.\nНа сложные ситуации — 1-2 предложения на языке онтологии.\"\"\"\n\nfor test in tests:\n    messages = [\n        {\"role\": \"system\", \"content\": SYSTEM_PROMPT},\n        {\"role\": \"user\", \"content\": test},\n    ]\n    inputs = tokenizer.apply_chat_template(messages, tokenize=True, add_generation_prompt=True, return_tensors=\"pt\")\n    outputs = model_b.generate(input_ids=inputs, max_new_tokens=80, temperature=0.2, do_sample=True)\n    response = tokenizer.decode(outputs[0][len(inputs[0]):], skip_special_tokens=True)\n    print(f\"\\nQ: {test[:60]}...\")\n    print(f\"A: {response.strip()}\")\n    print(\"---\")

## ЭКСПОРТ в GGUF для Orange Pi 5

In [ ]:
# Сохраняем LoRA адаптер\nmodel_b.save_pretrained(OUTPUT_DIR)\ntokenizer.save_pretrained(OUTPUT_DIR)\n\n# Экспорт в GGUF Q4_K_M (~400 MB)\nmodel_b.save_pretrained_gguf(\n    OUTPUT_DIR + \"-gguf\",\n    tokenizer,\n    quantization_method=\"q4_k_m\",\n)\n\nprint(f\"✓ GGUF saved to {OUTPUT_DIR}-gguf\")\nprint(\"  Copy to Orange Pi and run: ollama create serafim-social -f Modelfile\")

## Что мы ИЗОЛИРОВАЛИ\n\n| Слой | Проход A (domain) | Проход B (behavior) | Результат |\n|------|-------------------|---------------------|----------|\n| Attention Q,K,V,O | ЗАМОРОЖЕН | ФАЙНТЮНИНГ | Предпочтение дарить |\n| FFN gate,up,down | ФАЙНТЮНИНГ | ЗАМОРОЖЕН | Знание терминов |\n| Нижние слои (0-15) | ЗАМОРОЖЕН | ФАЙНТЮНИНГ | Русский язык сохранён |\n| Верхние слои (16-23) | ФАЙНТЮНИНГ | ФАЙНТЮНИНГ | Глубокое понимание |